## Data Ingestion 

In [1]:
## Document Structure

from langchain_core.documents import Document
import uuid

In [2]:
## Create a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [3]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print(" Sample text files created!")

 Sample text files created!


In [4]:
## READ THE TEXT FILES 

### TextLoader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)



/var/folders/hn/4yjl1t7x3kxgb9bs8nw8nxqm0000gn/T/ipykernel_19452/373965759.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/Users/godipally.shivakumar/Desktop/Projects/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [5]:
## ANOTHER WAY TO READ THE TEXT FILES

from langchain_community.document_loaders import DirectoryLoader 

## load all the text files in the directory 

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob ="**/*.txt",
    loader_cls= TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents 

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervise

In [6]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader , PyMuPDFLoader

pdf_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
)

pdf_documents = pdf_loader.load()

print(f"Loaded {len(pdf_documents)} PDF pages")

pdf_documents

Loaded 78 PDF pages


[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 16.1 (Windows)', 'creationdate': '2023-12-13T22:56:25+01:00', 'source': '../data/pdf/Photography_Book.pdf', 'file_path': '../data/pdf/Photography_Book.pdf', 'total_pages': 78, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-12-13T22:58:12+01:00', 'trapped': '', 'modDate': "D:20231213225812+01'00'", 'creationDate': "D:20231213225625+01'00'", 'page': 0}, page_content='1\n                         www.ianmiddletonphotography.com'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 16.1 (Windows)', 'creationdate': '2023-12-13T22:56:25+01:00', 'source': '../data/pdf/Photography_Book.pdf', 'file_path': '../data/pdf/Photography_Book.pdf', 'total_pages': 78, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-12-13T22:58:12+01:00', 'trapped': '', 'modDate': "D:20231213225812+01'00'", 'c

In [7]:
type(pdf_documents[0])

langchain_core.documents.base.Document

## Embedding and Vector DB 

In [8]:

import numpy as np
from sentence_transformers import SentenceTransformer 
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Any, Dict, Tuple
from sklearn.metrics.pairwise import cosine_similarity 


In [9]:
class EmbeddingManager :

    """This class is responsible for embeddings the documents using sentence-transformer archetiure"""

    def __init__(self, model_name :str = "all-MiniLM-L6-V2"):

        """Initialize the embedding manager 
        
        args:
            model_name : Huggingface model name for senetence embeddings"""
        
        self.model_name = model_name
        self.model =None
        self._load_model()
 

    def _load_model(self):
        """Load the sentence transformer model """
        try :
            print(f"Loading embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded successfully! Embeddings dimension : {self.model}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise      
 
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        
        Args:
        texts : List of text strings to embed
        
        Returns :
        numpy array of embeddings with shape (len(texts), embedding_dim)"""

        if not self.model:
            raise ValueError ("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts.....")
        embeddings = self.model.encode(texts, show_progress_bar= True)
        print(f"Generating embeddings with shape :{embeddings.shape}")
        return embeddings

## Initialize the EmbeddingManager

embedding_manager = EmbeddingManager()
embedding_manager 



Loading embedding model : all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6041.72it/s]


model loaded successfully! Embeddings dimension : SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


### VectorStore 

In [10]:
import os
import uuid
from typing import Any

import chromadb
import numpy as np


class VectorStore:
    """Manage document embeddings in ChromaDB."""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store",
    ):
        os.makedirs(persist_directory, exist_ok=True)

        # Open the existing database or create it if it does not exist
        self.client = chromadb.PersistentClient(
            path=persist_directory
        )

        # Open the existing collection or create it if it does not exist
        self.collection = self.client.get_or_create_collection(
            name=collection_name
        )

        print(f"Collection initialized: {collection_name}")
        print(f"Existing records: {self.collection.count()}")

    def add_documents(
        self,
        documents: list[Any],
        embeddings: np.ndarray,
    ):
        if len(documents) != len(embeddings):
            raise ValueError(
                "The number of documents and embeddings must match."
            )

        ids = []
        texts = []
        metadatas = []
        embedding_values = []

        for index, (document, embedding) in enumerate(
            zip(documents, embeddings)
        ):
            ids.append(f"chunk_{uuid.uuid4().hex}")

            texts.append(document.page_content)

            # Chroma supports string, integer, float and Boolean metadata
            metadata = {
                str(key): value
                for key, value in document.metadata.items()
                if isinstance(value, (str, int, float, bool))
            }
            metadata["chunk_index"] = index
            metadatas.append(metadata)

            embedding_values.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            documents=texts,
            embeddings=embedding_values,
            metadatas=metadatas,
        )

        print(f"Successfully stored {len(documents)} chunks.")
        print(f"Total records: {self.collection.count()}")


# Initialize the vector store
vector_store = VectorStore()

Collection initialized: pdf_documents
Existing records: 1686


### Chunking 

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
)

chunks = text_splitter.split_documents(pdf_documents)

print(f"PDF pages: {len(pdf_documents)}")
print(f"Chunks created: {len(chunks)}")

chunks

PDF pages: 78
Chunks created: 281


[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 16.1 (Windows)', 'creationdate': '2023-12-13T22:56:25+01:00', 'source': '../data/pdf/Photography_Book.pdf', 'file_path': '../data/pdf/Photography_Book.pdf', 'total_pages': 78, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-12-13T22:58:12+01:00', 'trapped': '', 'modDate': "D:20231213225812+01'00'", 'creationDate': "D:20231213225625+01'00'", 'page': 0}, page_content='1\n                         www.ianmiddletonphotography.com'),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 16.1 (Windows)', 'creationdate': '2023-12-13T22:56:25+01:00', 'source': '../data/pdf/Photography_Book.pdf', 'file_path': '../data/pdf/Photography_Book.pdf', 'total_pages': 78, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-12-13T22:58:12+01:00', 'trapped': '', 'modDate': "D:20231213225812+01'00'", 'c

In [12]:
# Extract text from the chunks
texts = [chunk.page_content for chunk in chunks]

# Generate an embedding for each chunk
embeddings = embedding_manager.generate_embeddings(texts)

print(f"Chunks: {len(chunks)}")
print(f"Embeddings: {len(embeddings)}")

# Store chunks in the existing ChromaDB collection
vector_store.add_documents(
    documents=chunks,
    embeddings=embeddings,
)

Generating embeddings for 281 texts.....


Batches: 100%|██████████| 9/9 [00:00<00:00, 11.54it/s]


Generating embeddings with shape :(281, 384)
Chunks: 281
Embeddings: 281
Successfully stored 281 chunks.
Total records: 1967


### Retriever pipeline from vectostore

In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store""" 

    def __init__ (self,vector_store,embedding_manager: EmbeddingManager):
        """Initialize the retriever
        Args:
        vector_store : Vector store containing document embeddings
        embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager